# Lab 11 - Research, Write & Fact-Check Multi-Agent Pipeline

**Estimated time:** 20min  ·  **Estimated cost:** ~$0.95

> **Multi-agent note:** this lab builds a coordinator that delegates to three specialists. Your account needs multi-agent enabled; request access if your key cannot create a coordinator roster.

![Architecture](assets/architecture.svg)

## Goal
Build a coordinator that delegates to three specialists and produces a verified, well-cited brief on any topic. A **Researcher** gathers sources with web tools, a **Writer** drafts a brief from those citations, and a **Fact-Checker** validates every claim against its linked source. The coordinator owns the session, fans work out to the roster, and assembles the final brief. You will watch the **primary thread** to see who did what, and learn how primary and session threads differ.

## Prerequisites
- the shared uv kernel `Managed Agents Labs (.venv)`
- An API key you can paste into the setup cell below. The notebook stores it as `ANTHROPIC_API_KEY` for this kernel session only.
- Multi-agent enabled on your account
- Comfort with the single-agent session loop from earlier labs

In [ ]:
# Verify that this notebook is using the uv-managed lab kernel.
import sys
from pathlib import Path

EXPECTED_KERNEL = "Managed Agents Labs (.venv)"
python_exe = Path(sys.executable)
python_prefix = Path(sys.prefix)

if ".venv" not in python_exe.parts and ".venv" not in python_prefix.parts:
    raise RuntimeError(
        f"Select the Jupyter kernel {EXPECTED_KERNEL!r} before running this notebook. "
        "From the repo root, run ./setup_uv.sh once to create and register it."
    )

print("Using uv environment:", python_prefix)

In [ ]:
import getpass
import os

if not os.environ.get("ANTHROPIC_API_KEY"):
    os.environ["ANTHROPIC_API_KEY"] = getpass.getpass("Enter your Anthropic API key: ")

print("ANTHROPIC_API_KEY configured for this notebook session.")

In [ ]:
import os
from pathlib import Path
from anthropic import Anthropic

# Udemy Labs note: the previous cell configures ANTHROPIC_API_KEY for this session.
assert os.environ.get("ANTHROPIC_API_KEY"), "Set ANTHROPIC_API_KEY first."

BETAS = ["managed-agents-2026-04-01"]
MODEL = os.environ.get("MODEL", "claude-haiku-4-5-20251001")  # course default; swap as models update
SUB_MODEL = "claude-haiku-4-5-20251001"  # course default model for the specialists
client = Anthropic()
print("SDK ready, coordinator model:", MODEL)

### Step 1 - Write the specialist prompts
Each specialist gets a tight job description. The researcher writes citations to a shared file, the writer reads them and drafts, the fact-checker validates the draft. All three share the same container filesystem, so they hand off artifacts via files in `/workspace`.

In [ ]:
RESEARCHER = """You research a topic. Use web_search for breadth and
web_fetch on the most promising results. Write a JSON array of citations
to /workspace/citations.json: each item {url, title, snippet, why_relevant}."""

WRITER = """You draft a brief from the researcher's citations.
Read /workspace/citations.json. Produce /workspace/draft.md (<=600 words)
with inline citation links. Do not invent claims."""

FACT_CHECKER = """For every claim in /workspace/draft.md, verify it against
the linked source via web_fetch. Write /workspace/check.md with one of:
  [verified]   - quote the source
  [partial]    - quote the source and explain
  [unverified] - explain why it failed."""

COORDINATOR = """You coordinate a research team. Given a topic:
1) Delegate to the researcher (return when /workspace/citations.json exists).
2) Delegate to the writer (return when /workspace/draft.md exists).
3) Delegate to the fact-checker (return when /workspace/check.md exists).
4) If check.md has any [unverified] claim, loop steps 2-3 with the writer
   fixing those claims.
5) When the draft is clean, save the final brief to
   /mnt/session/outputs/brief.md."""

print("prompts ready")

### Step 2 - Create the three specialists
The researcher and fact-checker need web tools, so they enable a focused subset of `agent_toolset_20260401`: the toolset is disabled by default and only the needed tools are turned on. All agents use the course default Haiku model; role differences come from prompts and tool scope.

In [ ]:
# The researcher needs the web tools to gather sources. Disable the toolset by
# default and enable only the tools this role needs.
researcher = client.beta.agents.create(
    name="Researcher", model=SUB_MODEL, system=RESEARCHER,
    tools=[{
        "type": "agent_toolset_20260401",
        "default_config": {"enabled": False},
        "configs": [
            {"name": "web_search", "enabled": True},
            {"name": "web_fetch",  "enabled": True},
            {"name": "write",      "enabled": True},
            {"name": "read",       "enabled": True},
        ],
    }],
    betas=BETAS,
)

# The writer only reads citations and writes a draft; the full toolset is fine.
writer = client.beta.agents.create(
    name="Writer", model=SUB_MODEL, system=WRITER,
    tools=[{"type": "agent_toolset_20260401"}],
    betas=BETAS,
)

# The fact-checker re-fetches each source to verify claims.
fact_checker = client.beta.agents.create(
    name="Fact-Checker", model=SUB_MODEL, system=FACT_CHECKER,
    tools=[{
        "type": "agent_toolset_20260401",
        "default_config": {"enabled": False},
        "configs": [
            {"name": "web_fetch", "enabled": True},
            {"name": "read",      "enabled": True},
            {"name": "write",     "enabled": True},
        ],
    }],
    betas=BETAS,
)
print("specialists:", researcher.id, writer.id, fact_checker.id)

### Step 3 - Configure `multiagent.agents` on the coordinator
The coordinator is the only agent your code talks to. It runs on the course default model because routing decisions matter most. Its roster lists the three specialists by id. A roster holds up to **20 unique agents** and is **1 level deep**: specialists cannot delegate further.

In [ ]:
coordinator = client.beta.agents.create(
    name="Research Lead", model=MODEL, system=COORDINATOR,
    tools=[{"type": "agent_toolset_20260401"}],
    multiagent={
        "type": "coordinator",
        "agents": [
            {"type": "agent", "id": researcher.id},
            {"type": "agent", "id": writer.id},
            {"type": "agent", "id": fact_checker.id},
        ],
    },
    betas=BETAS,
)
print("coordinator.id =", coordinator.id)

### Step 4 - Start a multiagent session
Create a cloud environment with networking so the web tools work, then open a session on the coordinator.

In [ ]:
# Unrestricted networking lets the web tools reach the internet.
env = client.beta.environments.create(
    name="multiagent-env",
    config={"type": "cloud", "networking": {"type": "unrestricted"}},
    betas=BETAS,
)

session = client.beta.sessions.create(
    agent={"type": "agent", "id": coordinator.id, "version": coordinator.version},
    environment_id=env.id,
    title="Brief: agentic AI state of the art",
    betas=BETAS,
)
print("session.id =", session.id)

### Step 5 - Send the topic and watch primary vs session threads
Stream the **primary thread**. It is a condensed view of all activity: one `session.thread_created` per specialist, an `agent.thread_message_received` each time a specialist returns, and the coordinator's own text. Drill into a specific session thread only when you need the full reasoning of one specialist.

In [ ]:
with client.beta.sessions.events.stream(session.id, betas=BETAS) as stream:
    client.beta.sessions.events.send(session.id, events=[{
        "type": "user.message",
        "content": [{
            "type": "text",
            "text": "Topic: 'agentic AI, state of the art'. Produce a verified brief.",
        }],
    }], betas=BETAS)
    for event in stream:
        if event.type == "session.thread_created":
            # A new child thread spawned for a specialist.
            print(f"\n+ thread {event.agent_name}")
        elif event.type == "agent.thread_message_received":
            # A specialist returned a result to the coordinator.
            print(f"  <- {event.from_agent_name} returned")
        elif event.type == "agent.message":
            # The coordinator's own text, including the final brief.
            for b in event.content:
                if b.type == "text":
                    print(b.text, end="", flush=True)
        elif event.type == "session.status_idle":
            print("\n--- session idle ---")
            break

### Step 6 - Collect the final brief
The coordinator saves `brief.md` into the session outputs. List the session files and download it.

In [ ]:
out_dir = Path("./outputs")
out_dir.mkdir(exist_ok=True)
for f in client.beta.files.list(scope_id=session.id, betas=BETAS):
    if f.filename == "brief.md":
        client.beta.files.download(f.id).write_to_file(str(out_dir / "brief.md"))
        print("saved:", out_dir / "brief.md")
        print("\n--- brief.md ---\n")
        print((out_dir / "brief.md").read_text())

### Cost estimate
Re-fetch the session id(s), print one list-price estimate per session, then print the total across all session ids. The estimate uses cumulative token usage plus Managed Agents active runtime; treat it as a course meter, not an invoice.


In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "shared"))
from cost_meter import print_sessions_cost

print_sessions_cost(client, [session.id], MODEL, betas=BETAS)


## Expected output
- A `session.id` line.
- Three threads spawn on the primary stream: `+ thread Researcher`, `+ thread Writer`, `+ thread Fact-Checker`.
- One `<- ... returned` line per specialist as each finishes.
- Optionally a second Writer and Fact-Checker pass if the first check flagged an unverified claim.
- A `brief.md` file: a researched, written, fact-checked brief with inline citations that survive verification.

```
session.id = sesn_01...

+ thread Researcher
  <- Researcher returned
+ thread Writer
  <- Writer returned
+ thread Fact-Checker
  <- Fact-Checker returned
--- session idle ---
saved: outputs/brief.md
```

## Troubleshooting
- **Roster rejected as too large** → a coordinator roster holds at most **20 unique agents**. This lab uses three, so if you hit the cap you have duplicate or stray roster entries.
- **A specialist tries to delegate and fails** → the roster is **1 level deep**. Specialists cannot have their own rosters or call other agents; only the coordinator delegates.
- **New delegations seem to stall** → a session allows **25 concurrent threads = 1 primary + 24 child**. Parallel fan-out can hit the ceiling; new delegations queue until a slot frees.
- **Slots stay full after work is done** → **archive** finished threads to return their slots to the budget. Idle child threads keep holding a slot until archived.
- **A specialist blocks on a tool you did not expect** → an `always_ask` tool confirmation is **cross-posted to the primary thread** with its `session_thread_id`. Reply on the **primary thread** and the server routes the confirmation back to the right child thread.
- **Coordinator skips a specialist** → tighten the coordinator prompt so each delegation step is explicit and names the file it should produce.